# 03 - Fault Tree Analysis: from logic to minimal cut sets

**Week 3; suggested duration: two teaching periods**

## Learning objectives

- Interpret AND/OR gates probabilistically
- Represent a fault tree in Python
- Build a recursive evaluator
- Interpret minimal cut sets and Birnbaum importance


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Fault-tree reasoning

**Which basic events or event combinations are sufficient for the top event?**

For independent inputs: AND gives $\prod p_i$, and OR gives $1-\prod(1-p_i)$. Independence is an explicit modeling assumption.


In [ ]:
def AND(*p): return float(np.prod(p))
def OR(*p): return 1-float(np.prod([1-x for x in p]))
p_lshh,p_xv=.01,.005
print("AND",AND(p_lshh,p_xv)); print("OR",OR(p_lshh,p_xv))


## 2. The first explicit fault tree

Start without a library: expose the complete logic in a Python `dict`.


In [ ]:
tree={"top_event":"AUTO_FAIL","nodes":{
 "AUTO_FAIL":{"type":"OR","children":["LSHH_FOD","XV_FOD"]},
 "LSHH_FOD":{"type":"BASIC","probability":.01},
 "XV_FOD":{"type":"BASIC","probability":.005}}}
tree


## 3. A recursive evaluator

Return the probability for a BASIC node. For a gate, evaluate its children before applying the gate operation.


In [ ]:
def evaluate(tree,node_id=None):
    node_id=node_id or tree["top_event"]
    node=tree["nodes"][node_id]
    typ=node["type"]
    if typ=="BASIC": return float(node["probability"])
    ps=[evaluate(tree,c) for c in node["children"]]
    if typ=="AND": return float(np.prod(ps))
    if typ=="OR": return 1-float(np.prod([1-p for p in ps]))
    raise ValueError(typ)
print(evaluate(tree))


## 4. The TK-101 overflow tree

Teaching model:

`OVERFLOW = HIGH_LEVEL_CHALLENGE AND AUTO_FAIL AND MANUAL_FAIL`

`AUTO_FAIL = LSHH_FOD OR XV_FOD`

`MANUAL_FAIL = LT101_LOW OR OPERATOR_MISS`


In [ ]:
tk={"top_event":"OVERFLOW","nodes":{
 "OVERFLOW":{"type":"AND","children":["CHALLENGE","AUTO_FAIL","MANUAL_FAIL"]},
 "AUTO_FAIL":{"type":"OR","children":["LSHH_FOD","XV_FOD"]},
 "MANUAL_FAIL":{"type":"OR","children":["LT101_LOW","OPERATOR_MISS"]},
 "CHALLENGE":{"type":"BASIC","probability":.08},
 "LSHH_FOD":{"type":"BASIC","probability":.01},
 "XV_FOD":{"type":"BASIC","probability":.005},
 "LT101_LOW":{"type":"BASIC","probability":.02},
 "OPERATOR_MISS":{"type":"BASIC","probability":.04}}}
print(f"P(OVERFLOW)={evaluate(tk):.8f}")


## 5. Minimal cut sets by hand and algorithmically

At an OR gate, collect the children's cut sets. At an AND gate, combine them. Remove duplicates and non-minimal supersets.


In [ ]:
from itertools import product
def cut_sets(tree,node_id=None):
    node_id=node_id or tree["top_event"]; node=tree["nodes"][node_id]
    if node["type"]=="BASIC": return [frozenset([node_id])]
    groups=[cut_sets(tree,c) for c in node["children"]]
    cand=[s for g in groups for s in g] if node["type"]=="OR" else [frozenset().union(*x) for x in product(*groups)]
    uniq=sorted(set(cand),key=lambda s:(len(s),sorted(s))); out=[]
    for s in uniq:
        if not any(x<=s for x in out): out.append(s)
    return out
for s in cut_sets(tk): print(sorted(s))


## 6. Birnbaum importance

Set a basic event to certainly failed (`p=1`), then to certainly functioning (`p=0`). The difference in top-event probability is the Birnbaum importance for the event under the stated model.


In [ ]:
from copy import deepcopy
def birnbaum(tree):
    out={}
    for e,n in tree["nodes"].items():
        if n["type"]!="BASIC": continue
        a,b=deepcopy(tree),deepcopy(tree)
        a["nodes"][e]["probability"]=1.; b["nodes"][e]["probability"]=0.
        out[e]=evaluate(a)-evaluate(b)
    return pd.Series(out).sort_values(ascending=False)
imp=birnbaum(tk); display(imp)
imp.plot(kind="barh"); plt.xlabel("Birnbaum importance"); plt.show()


## 7. Check with the shared `safetycourse.fta` module

After building the algorithm explicitly, use the shared module for automation and integration.


In [ ]:
from safetycourse.fta import top_event_probability,minimal_cut_sets
print(top_event_probability(tk))
print([sorted(x) for x in minimal_cut_sets(tk)])


## 8. Limitations

Static FTA does not naturally represent event ordering or time. Gate-input independence is not universal. Shared basic events and common-cause mechanisms require an appropriate dependence model; do not treat repeated occurrences of the same event as independent.


## Try it!

1. Set LT101_LOW to 0.10 and recalculate the top event.
2. Which basic event has the largest Birnbaum importance, and why?
3. Introduce COMMON_POWER_FAIL affecting both LSHH and XV. Explain why two independent copies are wrong.
4. Draw the four minimal cut sets by hand.


## Summary

FTA formalizes the combinational logic identified during qualitative analysis. Next, connect the **system model** to a fault tree through deterministic and verifiable transformations.
